# 01 — BiRefNet foreground extraction at original resolution

This notebook removes image **backgrounds** with BiRefNet for both datasets. Raw files are never overwritten. Each output contains:

- `images/`: original-resolution RGB foreground composited on white;
- `masks/`: matching grayscale foreground masks;
- `manifest.csv`: provenance, original size, output size, and status.

BiRefNet receives a temporary 1024×1024 tensor, but its mask is returned to the raw RGB dimensions. The saved RGB is never resized, padded, or upscaled. For each Industrial car, `images_raw/` and `images/` are compared and the folder with greater median pixel area is selected. Method-specific resizing happens in notebook 03.


In [ ]:
!pip -q install "transformers>=4.39" safetensors kornia timm


In [ ]:
import gc
import hashlib
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import kornia
import timm
import torch
from PIL import Image, ImageOps
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE != "cuda":
    raise RuntimeError("Connect a Colab GPU runtime before running BiRefNet. CPU would be unnecessarily slow.")
print("kornia:", kornia.__version__, "| timm:", timm.__version__)


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))

# Change only this value if the Drive project is moved.
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"

def find_unique_dir(names, search_roots):
    direct = [root / name for root in search_roots for name in names]
    matches = [p for p in direct if p.is_dir()]
    if not matches:
        for root in search_roots:
            if root.is_dir():
                matches.extend(p for p in root.rglob("*") if p.is_dir() and p.name.casefold() in {n.casefold() for n in names})
    unique = list(dict.fromkeys(p.resolve() for p in matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected one of {names}; found {len(unique)}: {unique}")
    return unique[0]

SEARCH_ROOTS = [PROJECT_ROOT / "data", PROJECT_ROOT]
INDUSTRIAL_ROOT = find_unique_dir(["IndustrialInventory"], SEARCH_ROOTS)
HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], SEARCH_ROOTS)

# If HQ200 is a wrapper folder, descend to the folder containing capture scenes.
if (HQ200_ROOT / "3DrealCarHQ200").is_dir():
    HQ200_ROOT = HQ200_ROOT / "3DrealCarHQ200"

print("Project:   ", PROJECT_ROOT)
print("Industrial:", INDUSTRIAL_ROOT)
print("3DRealCar: ", HQ200_ROOT)


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"

if not (CODE_ROOT / "code" / "src").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

CODE_PACKAGE_ROOT = CODE_ROOT / "code"
if str(CODE_PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_PACKAGE_ROOT))

print("Reusable code:", CODE_ROOT / "code" / "src")


In [ ]:
from src.image_preprocessing import collect_original_inventory, output_paths, summarize_resolutions


## Configuration

The default is a smoke test: three deterministic images from each dataset (six total). Smoke-test files are written below `_smoke_test` and cannot be confused with a complete run. When the results look correct, set `SMOKE_TEST=False` to process everything. Completed outputs are skipped, so rerunning after a Colab disconnect continues rather than starting over.


In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "data_processed" / "birefnet_original_resolution"
RUN_PREPROCESSING = True
SMOKE_TEST = True
SMOKE_IMAGES_PER_DATASET = 3
OVERWRITE = False
MODEL_ID = "ZhengPeng7/BiRefNet"
MODEL_INPUT_SIZE = (1024, 1024)
WHITE_BACKGROUND = (255, 255, 255)

inventory = collect_original_inventory(INDUSTRIAL_ROOT, HQ200_ROOT)
display(summarize_resolutions(inventory))
display(
    inventory.groupby(["dataset", "scene", "source_variant"])
    .agg(images=("source", "size"), width=("width", "first"), height=("height", "first"))
)
print("Industrial source folder selected per car by greatest median pixel area.")

def deterministic_sample(group, count):
    group = group.sort_values("source").reset_index(drop=True)
    if len(group) <= count:
        return group
    positions = np.linspace(0, len(group) - 1, count, dtype=int)
    return group.iloc[positions]

if SMOKE_TEST:
    work_inventory = pd.concat(
        [deterministic_sample(group, SMOKE_IMAGES_PER_DATASET) for _, group in inventory.groupby("dataset", sort=True)],
        ignore_index=True,
    )
    ACTIVE_OUTPUT_ROOT = OUTPUT_ROOT / "_smoke_test"
else:
    work_inventory = inventory.copy()
    ACTIVE_OUTPUT_ROOT = OUTPUT_ROOT

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}")
print(f"Images scheduled: {len(work_inventory):,}")
display(work_inventory[["dataset", "source", "width", "height"]])


In [ ]:
print("Output policy: foreground RGB and mask retain each raw source width and height.")
print("No crop, shared canvas, downscale, or upscale is applied in notebook 01.")


In [ ]:
transform_image = transforms.Compose([
    transforms.Resize(MODEL_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

model = None
if RUN_PREPROCESSING:
    model = AutoModelForImageSegmentation.from_pretrained(MODEL_ID, trust_remote_code=True)
    model.to(DEVICE).eval()
    if DEVICE == "cuda":
        model.half()
    print("Loaded", MODEL_ID)
else:
    print("Model not loaded. Set RUN_PREPROCESSING=True when ready.")


In [ ]:
def predict_mask(image):
    tensor = transform_image(image).unsqueeze(0).to(DEVICE)
    if DEVICE == "cuda":
        tensor = tensor.half()
    with torch.inference_mode():
        prediction = model(tensor)[-1].sigmoid()[0, 0].float().cpu().numpy()
    mask = Image.fromarray(np.uint8(np.clip(prediction, 0, 1) * 255), mode="L")
    return mask.resize(image.size, Image.Resampling.LANCZOS)

def destination_paths(row):
    source, dataset = row.source, row.dataset
    root = HQ200_ROOT if dataset == "3DRealCar" else INDUSTRIAL_ROOT
    image_path, mask_path = output_paths(ACTIVE_OUTPUT_ROOT, dataset, row.scene, source, root)
    return image_path, mask_path, row.scene

def process_one(row):
    output_image, output_mask, scene = destination_paths(row)
    if output_image.exists() and output_mask.exists() and not OVERWRITE:
        return {
            "status": "skipped", "scene": scene,
            "output_image": output_image, "output_mask": output_mask,
            "output_width": row.width, "output_height": row.height,
        }
    output_image.parent.mkdir(parents=True, exist_ok=True)
    output_mask.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(row.source) as opened:
        image = opened.convert("RGB")
    mask = predict_mask(image)
    foreground = Image.composite(image, Image.new("RGB", image.size, WHITE_BACKGROUND), mask)
    assert foreground.size == image.size and mask.size == image.size
    foreground.save(output_image, compress_level=3)
    mask.save(output_mask, compress_level=3)
    return {
        "status": "written", "scene": scene,
        "output_image": output_image, "output_mask": output_mask,
        "output_width": image.width, "output_height": image.height,
    }


In [ ]:
results = []
if RUN_PREPROCESSING:
    for number, row in enumerate(work_inventory.itertuples(index=False), start=1):
        try:
            result = process_one(row)
            result.update({
                "dataset": row.dataset, "source": row.source,
                "source_variant": row.source_variant,
                "source_width": row.width, "source_height": row.height,
            })
        except Exception as error:
            result = {"dataset": row.dataset, "source": row.source, "status": "error", "error": repr(error)}
        results.append(result)
        if number % 25 == 0 or number == len(work_inventory):
            print(f"{number:,}/{len(work_inventory):,}", Counter(r["status"] for r in results))
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    manifest = pd.DataFrame(results)
    ACTIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(ACTIVE_OUTPUT_ROOT / "manifest.csv", index=False)
    complete = manifest.status.isin(["written", "skipped"])
    assert (manifest.loc[complete, "source_width"] == manifest.loc[complete, "output_width"]).all()
    assert (manifest.loc[complete, "source_height"] == manifest.loc[complete, "output_height"]).all()
    display(manifest.groupby(["dataset", "status"]).size().rename("images"))
    print("Manifest:", ACTIVE_OUTPUT_ROOT / "manifest.csv")
else:
    print(f"Dry run only: {len(work_inventory):,} images scheduled. Set RUN_PREPROCESSING=True to start/resume.")


In [ ]:
if RUN_PREPROCESSING and results:
    successful = pd.DataFrame(results)
    successful = successful[successful.status.isin(["written", "skipped"])]
    successful = pd.concat([
        deterministic_sample(group, 3) for _, group in successful.groupby("dataset", sort=True)
    ], ignore_index=True)
    fig, axes = plt.subplots(len(successful), 3, figsize=(13, 4 * len(successful)), squeeze=False)
    for row_axes, (_, row) in zip(axes, successful.iterrows()):
        with Image.open(row.source) as source_image:
            row_axes[0].imshow(source_image.convert("RGB"))
        row_axes[1].imshow(Image.open(row.output_image).convert("RGB"))
        row_axes[2].imshow(Image.open(row.output_mask).convert("L"), cmap="gray", vmin=0, vmax=255)
        row_axes[0].set_ylabel(row.dataset)
        for ax, title in zip(row_axes, ["Original", "BiRefNet foreground", "Mask"]):
            ax.set_title(title)
            ax.axis("off")
    plt.tight_layout()
    plt.show()
    print("These are at most three examples per dataset. If they look correct, set SMOKE_TEST=False for the full run.")


## Important reconstruction note

BiRefNet masks the object but does not improve camera poses or add image detail. This notebook preserves the original pixel grid, so it does not require an intrinsic-coordinate change. Notebook 03 records every later crop, scale, and padding offset needed to transform known intrinsics consistently.
